# Lab 2 — Metrics, baselines và cost-sensitive operating point

**Thời lượng:** 105 phút · **Case:** support triage · **Mục tiêu:** không để accuracy và test leakage dẫn dắt quyết định.

## Protocol khóa trước khi xem kết quả

- Sort theo `created_at`, split 70/15/15.
- Baseline: majority class và `triage_risk_score` hiện hành.
- Validation grid: 0.05 đến 0.95, bước 0.05.
- Cost giả định: FP=10, FN=200.
- Tie-break: cost nhỏ nhất → recall cao hơn → threshold thấp hơn.
- Sau khi threshold khóa, test chỉ mở một lần.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
OUTPUT = ROOT / 'outputs'
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA / 'support_triage.csv')
print(df.shape)
display(df.head(3))

## 1. Chronological split

Random split có thể làm tương lai rò vào quá khứ và không mô phỏng deployment. Dùng stable sort để xử lý timestamp bằng nhau nhất quán.

In [ ]:
def chronological_indices(timestamps, train_fraction=0.70, validation_fraction=0.15):
    times = pd.to_datetime(pd.Series(timestamps), errors='raise').to_numpy()
    order = np.argsort(times, kind='stable')
    n = len(order)
    n_train = int(np.floor(n * train_fraction))
    n_validation = int(np.floor(n * validation_fraction))
    return order[:n_train], order[n_train:n_train+n_validation], order[n_train+n_validation:]

train_idx, validation_idx, test_idx = chronological_indices(df['created_at'])
assert (len(train_idx), len(validation_idx), len(test_idx)) == (503, 108, 109)
assert pd.to_datetime(df.iloc[train_idx]['created_at']).max() <= pd.to_datetime(df.iloc[validation_idx]['created_at']).min()
assert pd.to_datetime(df.iloc[validation_idx]['created_at']).max() <= pd.to_datetime(df.iloc[test_idx]['created_at']).min()
print('Split sizes:', len(train_idx), len(validation_idx), len(test_idx))

In [ ]:
TARGET = 'escalated_within_48h'
SCORE = 'triage_risk_score'
prevalence = {
    'train': float(df.iloc[train_idx][TARGET].mean()),
    'validation': float(df.iloc[validation_idx][TARGET].mean()),
    'test_locked': 'not opened until policy is frozen',
}
prevalence

## 2. Confusion metrics và dummy baseline

Accuracy phải được đọc cùng confusion matrix. Với positive hiếm, majority-negative có accuracy cao nhưng recall bằng 0.

In [ ]:
def binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'accuracy': float((tp + tn) / len(y_true)),
        'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
    }

def majority_baseline(y_train, y_eval):
    counts = np.bincount(np.asarray(y_train, dtype=int), minlength=2)
    majority = int(np.argmax(counts))
    return {'majority_class': majority, **binary_metrics(y_eval, np.full(len(y_eval), majority))}

validation_majority = majority_baseline(
    df.iloc[train_idx][TARGET], df.iloc[validation_idx][TARGET]
)
validation_majority

## 3. Validation threshold sweep

`TotalCost = 10×FP + 200×FN`. Đây là cost proxy của bài học, không phải giá trị đã được doanh nghiệp phê duyệt.

In [ ]:
FP_COST = 10.0
FN_COST = 200.0
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)

def threshold_sweep(y_true, scores, threshold_grid):
    rows = []
    for threshold in threshold_grid:
        metrics = binary_metrics(y_true, np.asarray(scores) >= threshold)
        rows.append({
            'threshold': float(threshold),
            **metrics,
            'total_cost': float(metrics['fp'] * FP_COST + metrics['fn'] * FN_COST),
        })
    return pd.DataFrame(rows)

validation = df.iloc[validation_idx]
sweep = threshold_sweep(validation[TARGET], validation[SCORE], thresholds)
display(sweep.sort_values(['total_cost', 'recall', 'threshold'], ascending=[True, False, True]).head(8))

In [ ]:
ordered = sweep.sort_values(
    ['total_cost', 'recall', 'threshold'],
    ascending=[True, False, True],
    kind='stable',
)
selected_threshold = float(ordered.iloc[0]['threshold'])
selected_validation = ordered.iloc[0].to_dict()
assert selected_threshold == 0.05
assert selected_validation['total_cost'] == 1120.0
print('LOCKED threshold:', selected_threshold)
selected_validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(sweep['threshold'], sweep['total_cost'], marker='o')
axes[0].axvline(selected_threshold, color='crimson', linestyle='--', label='selected')
axes[0].set(xlabel='Threshold', ylabel='Validation cost', title='Cost by operating point')
axes[0].legend()
axes[1].plot(sweep['threshold'], sweep['precision'], marker='o', label='precision')
axes[1].plot(sweep['threshold'], sweep['recall'], marker='o', label='recall')
axes[1].set(xlabel='Threshold', ylabel='Metric', ylim=(-0.02, 1.02), title='Precision–recall trade-off')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / 'support_threshold_tradeoff.png', dpi=150)
plt.show()

## 4. Final test — mở đúng một lần

Từ cell này trở đi không thay threshold/grid/cost. Nếu thay đổi, kết quả test phải được đánh dấu exploratory và cần holdout mới.

In [ ]:
test_evaluation_count = 0
test = df.iloc[test_idx].copy()
test['prediction'] = (test[SCORE].to_numpy() >= selected_threshold).astype(int)
test_metrics = binary_metrics(test[TARGET], test['prediction'])
test_metrics['total_cost'] = float(
    test_metrics['fp'] * FP_COST + test_metrics['fn'] * FN_COST
)
test_evaluation_count += 1
majority_test = majority_baseline(df.iloc[train_idx][TARGET], test[TARGET])
majority_test['total_cost'] = float(
    majority_test['fp'] * FP_COST + majority_test['fn'] * FN_COST
)
assert test_evaluation_count == 1
print('Heuristic test:', test_metrics)
print('Majority test:', majority_test)

In [ ]:
def slice_metrics(frame, columns):
    rows = []
    for column in columns:
        for value, group in frame.groupby(column, dropna=False):
            metrics = binary_metrics(group[TARGET], group['prediction'])
            rows.append({
                'slice_column': column, 'slice_value': value, 'n': len(group),
                'prevalence': float(group[TARGET].mean()),
                'precision': metrics['precision'], 'recall': metrics['recall'],
                'predicted_positive_rate': float(group['prediction'].mean()),
                'total_cost': metrics['fp'] * FP_COST + metrics['fn'] * FN_COST,
            })
    return pd.DataFrame(rows)

slices = slice_metrics(test, ['channel', 'customer_tier'])
display(slices)

In [ ]:
sweep.to_csv(OUTPUT / 'support_threshold_sweep.csv', index=False)
slices.to_csv(OUTPUT / 'support_slice_metrics.csv', index=False)
summary = {
    'split_sizes': {'train': len(train_idx), 'validation': len(validation_idx), 'test': len(test_idx)},
    'prevalence': {
        'train': float(df.iloc[train_idx][TARGET].mean()),
        'validation': float(df.iloc[validation_idx][TARGET].mean()),
        'test': float(test[TARGET].mean()),
    },
    'selected_threshold': selected_threshold,
    **test_metrics,
    'predicted_positive_rate': float(test['prediction'].mean()),
    'test_evaluation_count': test_evaluation_count,
    'majority_test_baseline': majority_test,
}
with (OUTPUT / 'support_test_metrics.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)
summary

In [ ]:
assert selected_threshold == 0.05
assert {key: test_metrics[key] for key in ['tp', 'fp', 'tn', 'fn']} == {
    'tp': 14, 'fp': 76, 'tn': 17, 'fn': 2
}
assert np.isclose(test_metrics['recall'], 0.875)
assert test_metrics['total_cost'] == 1160.0
assert majority_test['accuracy'] > test_metrics['accuracy']
assert majority_test['recall'] == 0.0
assert majority_test['total_cost'] == 3200.0
assert summary['predicted_positive_rate'] > 0.20
assert (FIGURES / 'support_threshold_tradeoff.png').exists()
print('PASS — reference metrics reproduced and test opened exactly once.')

## Diễn giải và exit ticket

1. Majority baseline có accuracy cao nhưng recall=0 và cost=3,200; heuristic cost=1,160 vì FN đắt. Vì vậy accuracy không phải primary decision metric.  
2. Threshold 0.05 cho recall cao nhưng predicted-positive rate vượt 20% capacity. Kết luận phải là **Conditional Go/No-Go ở constraint hiện tại**, không phải production-ready.  
3. Viết hai thay đổi: một thay đổi cost ratio và một thay đổi capacity; dự đoán threshold/policy sẽ dịch theo hướng nào trước khi thử.